# ADS 层 — KPI 指标

从 DWS 聚合产出核心业务指标：
- **单量（trip_count）**：各来源、各月总行程数
- **GMV**：总流水金额
- **平台抽成率（surcharge_rate）**：附加费（市政税 + 改善费 + 拥堵费 + 机场费）占票价比例
- **小费率（tip_rate）**：小费 / 票价
- **人均里程、人均时长** 等效率指标

## 1. SparkSession

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-ADS-Metrics")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE    = os.path.expanduser("~/tlc-spark-pipeline")
DWS_IN  = f"{BASE}/data/warehouse/dws/trips_daily"
ADS_OUT = f"{BASE}/data/warehouse/ads/kpi_daily"
print(f"Spark {spark.version} ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 00:00:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1 ready


## 2. 月度 KPI 汇总

In [2]:
dws = spark.read.parquet(DWS_IN)

monthly_kpi = (
    dws
    .groupBy("pickup_year", "pickup_month", "source_type")
    .agg(
        F.sum("trip_count").alias("trip_count"),
        F.round(F.sum("gmv"), 2).alias("gmv"),
        F.round(F.sum("fare_sum"), 2).alias("fare_sum"),
        F.round(F.sum("tip_sum"), 2).alias("tip_sum"),
        F.round(F.sum("surcharge_sum"), 2).alias("surcharge_sum"),
        F.round(F.sum("distance_sum"), 2).alias("distance_sum"),
        F.round(F.sum("duration_sum"), 2).alias("duration_sum")
    )
    # 平台抽成率 = 附加费 / 票价（市政税+改善费+拥堵费+机场费 占 fare 的比例）
    .withColumn("surcharge_rate",
        F.round(F.col("surcharge_sum") / F.col("fare_sum"), 4))
    # 小费率
    .withColumn("tip_rate",
        F.round(F.col("tip_sum") / F.col("fare_sum"), 4))
    # 均值指标
    .withColumn("avg_fare",
        F.round(F.col("fare_sum") / F.col("trip_count"), 2))
    .withColumn("avg_distance_mi",
        F.round(F.col("distance_sum") / F.col("trip_count"), 2))
    .withColumn("avg_duration_min",
        F.round(F.col("duration_sum") / F.col("trip_count"), 1))
    .orderBy("pickup_year", "pickup_month", "source_type")
)

print("=== 月度 KPI ===")
monthly_kpi.select(
    "pickup_month", "source_type", "trip_count", "gmv",
    "surcharge_rate", "tip_rate", "avg_fare"
).show(20)

=== 月度 KPI ===


+------------+-----------+----------+-------------+--------------+--------+--------+
|pickup_month|source_type|trip_count|          gmv|surcharge_rate|tip_rate|avg_fare|
+------------+-----------+----------+-------------+--------------+--------+--------+
|           1|      green|     48991|   1058578.18|         0.152|   0.146|   15.93|
|           1|     yellow|   2713403|7.425501547E7|        0.2158|  0.1887|   18.39|
|           2|      green|     46581|   1026241.08|        0.1481|  0.1465|   16.26|
|           2|     yellow|   2709721|7.409687263E7|        0.2164|  0.1907|   18.34|
|           3|      green|     51018|   1144585.77|        0.1411|  0.1439|   16.71|
|           3|     yellow|   3024319|8.559223021E7|        0.2079|  0.1882|   19.11|
+------------+-----------+----------+-------------+--------------+--------+--------+



## 3. 写出 ADS

In [3]:
%%time
monthly_kpi.write.mode("overwrite").parquet(ADS_OUT)
print(f"ADS written to: {ADS_OUT}")

# 输出汇总供简历填写
print("\n=== 全量汇总 ===")
total = monthly_kpi.groupBy("source_type").agg(
    F.sum("trip_count").alias("total_trips"),
    F.round(F.sum("gmv"), 0).alias("total_gmv"),
    F.round(F.avg("surcharge_rate") * 100, 1).alias("avg_surcharge_rate_pct"),
    F.round(F.avg("tip_rate") * 100, 1).alias("avg_tip_rate_pct")
)
total.show()

spark.stop()
print("ADS metrics complete.")

ADS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/ads/kpi_daily

=== 全量汇总 ===


+-----------+-----------+------------+----------------------+----------------+
|source_type|total_trips|   total_gmv|avg_surcharge_rate_pct|avg_tip_rate_pct|
+-----------+-----------+------------+----------------------+----------------+
|      green|     146590|   3229405.0|                  14.7|            14.5|
|     yellow|    8447443|2.33944118E8|                  21.3|            18.9|
+-----------+-----------+------------+----------------------+----------------+



ADS metrics complete.
CPU times: user 14.7 ms, sys: 9.86 ms, total: 24.5 ms
Wall time: 2.98 s
